In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS quickcart.quarantine;
CREATE SCHEMA IF NOT EXISTS quickcart.silver;

In [0]:
%sql
SHOW SCHEMAS IN quickcart;

### Creating Silver Audit Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS quickcart.silver.silver_audit (
    run_id STRING,
    table_name STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    status STRING,
    source_count BIGINT,
    valid_count BIGINT,
    invalid_count BIGINT,
    duplicate_count BIGINT,
    silver_count BIGINT,
    error_message STRING
)
USING DELTA;

In [0]:
# %sql
# ALTER TABLE quickcart.silver.silver_audit
# ADD COLUMNS (
#     watermark_value TIMESTAMP);

In [0]:
spark.table(
    "quickcart.silver.silver_audit"
).printSchema()

###Import required libraries

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import uuid
from datetime import datetime
from pyspark.sql.types import *
from delta.tables import DeltaTable

#####Data Profiling

In [0]:
# customers_bronze = spark.table("""
#                             quickcart.bronze.customers
#                             """)

# #customer_bronze.printSchema()

# print("Bronze Customers:", customers_bronze.count())

#Checking NULLS

# customers_bronze.select([
#     F.count(F.when(F.col(c).isNull(), c)).alias(c) 
#     for c in customers_bronze.columns]).display()

#Duplicate records

# customers_bronze.groupBy(col("customer_id")).agg(count(col("customer_id")).alias("c"))\
#     .filter(col("c")>1).display()

#Check invalid customer IDs

#customers_bronze.filter(col("customer_id").isNull()).count()
#customers_bronze.filter(trim(col("customer_id")) == "").count()

#Check email quality

# customers_bronze.filter(
#     ~F.col("email").rlike(
#         r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
#     )
# ).select(
#     "customer_id",
#     "email"
# ).display()



###Silver and Validation Rules Config

In [0]:
SILVER_CONFIG = {
    "customers" : {
        "bronze_table" : "quickcart.bronze.customers",
        "silver_table" : "quickcart.silver.customers",
        "quarantine_table" : "quickcart.quarantine.customers",
        "primary_key" : "customer_id",
        "watermark_column" : "updated_at",
    },
    "products" : {
        "bronze_table" : "quickcart.bronze.products",
        "silver_table" : "quickcart.silver.products",
        "quarantine_table" : "quickcart.quarantine.products",
        "primary_key" : "product_id",
        "watermark_column" : "updated_at",
    },
    "orders" : {
        "bronze_table" : "quickcart.bronze.orders",
        "silver_table" : "quickcart.silver.orders",
        "quarantine_table" : "quickcart.quarantine.orders",
        "primary_key" : "order_id",
        "watermark_column" : "updated_at",
    },
    "payments" : {
        "bronze_table" : "quickcart.bronze.payments",
        "silver_table" : "quickcart.silver.payments",
        "quarantine_table" : "quickcart.quarantine.payments",
        "primary_key" : "payment_id",
        "watermark_column" : "updated_at",
    },
    "deliveries" : {
        "bronze_table" : "quickcart.bronze.deliveries",
        "silver_table" : "quickcart.silver.deliveries",
        "quarantine_table" : "quickcart.quarantine.deliveries",
        "primary_key" : "delivery_id",
        "watermark_column" : "updated_at",
    }
}

In [0]:
VALIDATION_RULES = {

    # ============================================================
    # CUSTOMERS
    # ============================================================

    "customers": {

        "customer_id": {
            "condition": (
                F.col("customer_id").isNull() |
                (F.trim(F.col("customer_id")) == "")
            ),
            "error_code": "INVALID_CUSTOMER_ID"
        },

        "email": {
            "condition": (
                F.col("email").isNotNull() &
                ~F.col("email").rlike(
                    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
                )
            ),
            "error_code": "INVALID_EMAIL"
        },

        "gender": {
            "condition": (
                F.col("gender").isNotNull() &
                ~F.col("gender").isin(
                    "M",
                    "F",
                    "MALE",
                    "FEMALE",
                    "OTHER"
                )
            ),
            "error_code": "INVALID_GENDER"
        }
    },


    # ============================================================
    # PRODUCTS
    # ============================================================

    "products": {

        "product_id": {
            "condition": (
                F.col("product_id").isNull() |
                (F.trim(F.col("product_id")) == "")
            ),
            "error_code": "INVALID_PRODUCT_ID"
        },

        "product_name": {
            "condition": (
                F.col("product_name").isNull() |
                (F.trim(F.col("product_name")) == "")
            ),
            "error_code": "INVALID_PRODUCT_NAME"
        },

        "category": {
            "condition": (
                F.col("category").isNull() |
                (F.trim(F.col("category")) == "")
            ),
            "error_code": "INVALID_CATEGORY"
        },

        "created_date": {
            "condition": F.col("created_date").isNull(),
            "error_code": "INVALID_CREATED_DATE"
        },

        "updated_at": {
            "condition": F.col("updated_at").isNull(),
            "error_code": "INVALID_UPDATED_AT"
        },

        "price": {
            "condition": (
                F.col("price").isNull() |
                (F.col("price") <= 0)
            ),
            "error_code": "INVALID_PRICE"
        },

        "cost": {
            "condition": (
                F.col("cost").isNull() |
                (F.col("cost") < 0)
            ),
            "error_code": "INVALID_COST"
        },

        "cost_vs_price": {
            "condition": (
                F.col("cost").isNotNull() &
                F.col("price").isNotNull() &
                (F.col("cost") > F.col("price"))
            ),
            "error_code": "INVALID_COST_PRICE"
        },

        "product_rating": {
            "condition": (
                F.col("product_rating").isNotNull() &
                (
                    (F.col("product_rating") < 0) |
                    (F.col("product_rating") > 5)
                )
            ),
            "error_code": "INVALID_PRODUCT_RATING"
        }
    },


    # ============================================================
    # ORDERS
    # ============================================================

    "orders": {

        "order_id": {
            "condition": (
                F.col("order_id").isNull() |
                (F.trim(F.col("order_id")) == "")
            ),
            "error_code": "INVALID_ORDER_ID"
        },

        "customer_id": {
            "condition": (
                F.col("customer_id").isNull() |
                (F.trim(F.col("customer_id")) == "")
            ),
            "error_code": "INVALID_CUSTOMER_ID"
        },

        "product_id": {
            "condition": (
                F.col("product_id").isNull() |
                (F.trim(F.col("product_id")) == "")
            ),
            "error_code": "INVALID_PRODUCT_ID"
        },

        "quantity": {
            "condition": (
                F.col("quantity").isNull() |
                (F.col("quantity") <= 0)
            ),
            "error_code": "INVALID_QUANTITY"
        },

        "price": {
            "condition": (
                F.col("price").isNull() |
                (F.col("price") <= 0)
            ),
            "error_code": "INVALID_PRICE"
        },

        "discount_percentage": {
            "condition": (
                F.col("discount_percentage").isNotNull() &
                (
                    (F.col("discount_percentage") < 0) |
                    (F.col("discount_percentage") > 100)
                )
            ),
            "error_code": "INVALID_DISCOUNT_PERCENTAGE"
        },

        "discount_amount": {
            "condition": (
                F.col("discount_amount").isNull() |
                (F.col("discount_amount") < 0)
            ),
            "error_code": "INVALID_DISCOUNT_AMOUNT"
        },

        "order_amount": {
            "condition": (
                F.col("order_amount").isNull() |
                (F.col("order_amount") < 0)
            ),
            "error_code": "INVALID_ORDER_AMOUNT"
        },

        "payment_method": {
            "condition": (
                F.col("payment_method").isNull() |
                ~F.col("payment_method").isin(
                    "COD",
                    "Credit Card",
                    "Wallet",
                    "Net Banking",
                    "Debit Card",
                    "UPI"
                )
            ),
            "error_code": "INVALID_PAYMENT_METHOD"
        },

        "order_status": {
            "condition": (
                F.col("order_status").isNull() |
                ~F.col("order_status").isin(
                    "DELIVERED",
                    "CANCELLED",
                    "CONFIRMED",
                    "RETURNED",
                    "PLACED",
                    "SHIPPED"
                )
            ),
            "error_code": "INVALID_ORDER_STATUS"
        },

        "order_timestamp": {
            "condition": F.col("order_timestamp").isNull(),
            "error_code": "INVALID_ORDER_TIMESTAMP"
        },

        "updated_at": {
            "condition": F.col("updated_at").isNull(),
            "error_code": "INVALID_UPDATED_AT"
        }
    },


    # ============================================================
    # PAYMENTS
    # ============================================================

    "payments": {

        "payment_id": {
            "condition": (
                F.col("payment_id").isNull() |
                (F.trim(F.col("payment_id")) == "")
            ),
            "error_code": "INVALID_PAYMENT_ID"
        },

        "order_id": {
            "condition": (
                F.col("order_id").isNull() |
                (F.trim(F.col("order_id")) == "")
            ),
            "error_code": "INVALID_ORDER_ID"
        },

        "payment_method": {
            "condition": (
                F.col("payment_method").isNull() |
                ~F.col("payment_method").isin(
                    "COD",
                    "Credit Card",
                    "Wallet",
                    "Net Banking",
                    "Debit Card",
                    "UPI"
                )
            ),
            "error_code": "INVALID_PAYMENT_METHOD"
        },

        "payment_status": {
            "condition": (
                F.col("payment_status").isNull() |
                ~F.col("payment_status").isin(
                    "PENDING",
                    "FAILED",
                    "SUCCESS",
                    "REFUNDED"
                )
            ),
            "error_code": "INVALID_PAYMENT_STATUS"
        },

        "transaction_amount": {
            "condition": (
                F.col("transaction_amount").isNull() |
                (F.col("transaction_amount") <= 0)
            ),
            "error_code": "INVALID_TRANSACTION_AMOUNT"
        },

        "transaction_timestamp": {
            "condition": F.col("transaction_timestamp").isNull(),
            "error_code": "INVALID_TRANSACTION_TIMESTAMP"
        },

        "transaction_reference": {
            "condition": (
                F.col("transaction_reference").isNull() |
                (F.trim(F.col("transaction_reference")) == "")
            ),
            "error_code": "INVALID_TRANSACTION_REFERENCE"
        },

        "updated_at": {
            "condition": F.col("updated_at").isNull(),
            "error_code": "INVALID_UPDATED_AT"
        }
    },


    # ============================================================
    # DELIVERIES
    # ============================================================

    "deliveries": {

        "delivery_id": {
            "condition": (
                F.col("delivery_id").isNull() |
                (F.trim(F.col("delivery_id")) == "")
            ),
            "error_code": "INVALID_DELIVERY_ID"
        },

        "order_id": {
            "condition": (
                F.col("order_id").isNull() |
                (F.trim(F.col("order_id")) == "")
            ),
            "error_code": "INVALID_ORDER_ID"
        },

        "delivery_partner": {
            "condition": (
                F.col("delivery_partner").isNull() |
                ~F.col("delivery_partner").isin(
                    "BlueDart",
                    "FastTrack",
                    "Delhivery",
                    "EcomExpress",
                    "QuickShip"
                )
            ),
            "error_code": "INVALID_DELIVERY_PARTNER"
        },

        "warehouse": {
            "condition": (
                F.col("warehouse").isNull() |
                (F.trim(F.col("warehouse")) == "")
            ),
            "error_code": "INVALID_WAREHOUSE"
        },

        "shipping_city": {
            "condition": (
                F.col("shipping_city").isNull() |
                (F.trim(F.col("shipping_city")) == "")
            ),
            "error_code": "INVALID_SHIPPING_CITY"
        },

        "delivery_status": {
            "condition": (
                F.col("delivery_status").isNull() |
                ~F.col("delivery_status").isin(
                    "RETURNED",
                    "PROCESSING",
                    "DELIVERED",
                    "IN_TRANSIT"
                )
            ),
            "error_code": "INVALID_DELIVERY_STATUS"
        },

        "order_timestamp": {
            "condition": F.col("order_timestamp").isNull(),
            "error_code": "INVALID_ORDER_TIMESTAMP"
        },

        "delivery_attempts": {
            "condition": (
                F.col("delivery_attempts").isNull() |
                (F.col("delivery_attempts") < 0)
            ),
            "error_code": "INVALID_DELIVERY_ATTEMPTS"
        },

        "updated_at": {
            "condition": F.col("updated_at").isNull(),
            "error_code": "INVALID_UPDATED_AT"
        },

        "shipped_date": {
            "condition": (
                F.col("shipped_date").isNotNull() &
                F.col("order_timestamp").isNotNull() &
                (F.col("shipped_date") < F.col("order_timestamp"))
            ),
            "error_code": "INVALID_SHIPPED_DATE"
        },

        "estimated_delivery_date": {
            "condition": (
                F.col("estimated_delivery_date").isNotNull() &
                F.col("shipped_date").isNotNull() &
                (
                    F.col("estimated_delivery_date")
                    < F.col("shipped_date")
                )
            ),
            "error_code": "INVALID_ESTIMATED_DELIVERY_DATE"
        },

        "actual_delivery_date": {
            "condition": (
                F.col("actual_delivery_date").isNotNull() &
                F.col("shipped_date").isNotNull() &
                (
                    F.col("actual_delivery_date")
                    < F.col("shipped_date")
                )
            ),
            "error_code": "INVALID_ACTUAL_DELIVERY_DATE"
        }
    }
}

###Test the configuration

In [0]:
config = SILVER_CONFIG['customers']
valid_config = VALIDATION_RULES['customers']

for rule_name in valid_config.keys():
    print(rule_name)
    


# for tables in SILVER_CONFIG:
#     print(tables)

###Table validation function

In [0]:
def validate_table_config(table_name):
    if table_name not in SILVER_CONFIG:
        raise ValueError(
            f"Unsupported table: {table_name}. "
            f"Supported tables: {list(SILVER_CONFIG.keys())}"
            ) 
    return SILVER_CONFIG[table_name]

#print(validate_table_config("customers"))

### Getting Baseline Watermark

In [0]:
baseline_watermark = spark.table("quickcart.bronze.customers")\
    .agg(max("updated_at").alias("max_updated_at"))\
        .first()["max_updated_at"]

print(baseline_watermark)

###Getting last watermark

In [0]:
def get_last_watermark(table_name):
    audit_table = "quickcart.silver.silver_audit"
    result = (
        spark.table(audit_table)
        .filter(
            (F.col("table_name") == table_name) &
            (F.col("status") == "SUCCESS") &
            F.col("watermark_value").isNotNull()
        )
        .agg(
            F.max("watermark_value").alias("last_watermark")
        )
        .collect()
    )

    if not result:
        return None

    return result[0]["last_watermark"]

### Get incremented data from Bronze

In [0]:
def get_incremental_data(table_name, bronze_df):
    config = validate_table_config(table_name)
    watermark_column = config["watermark_column"]

    last_watermark = get_last_watermark(table_name)

    if last_watermark is None:
        print("No previous watermark found.")
        print("Processing full Bronze dataset.")
        return bronze_df
    else:
        incremental_df = bronze_df.filter(col(watermark_column) > F.lit(last_watermark))
        return incremental_df

###Standardization function

In [0]:
def standardize_columns(table_name, df):
    if table_name == "customers":
        df = df\
            .withColumn("customer_id", F.trim(F.col("customer_id")))\
                .withColumn("customer_name", F.trim(F.col("customer_name")))\
                    .withColumn("email", F.lower(F.trim(F.col("email"))))\
                        .withColumn("phone", F.regexp_replace(F.trim(F.col("phone")),r"\s+",""))\
                            .withColumn("gender", F.upper(F.trim(F.col("gender"))))\
                                .withColumn("city", F.initcap(F.trim(F.col("city"))))\
                                    .withColumn("state", F.initcap(F.trim(F.col("state"))))\
                                        .withColumn("pincode", F.trim(F.col("pincode")))\
                                            .withColumn("customer_segment", F.upper(F.trim(F.col("customer_name"))))
    elif table_name == "products":
        pass
    elif table_name == "orders":
        pass
    elif table_name == "payments":
        pass
    elif table_name == "deliveries":
        pass
    return df

###Reusable validation function

In [0]:
# def apply_validation_rules(table_name, df):
#     if table_name not in VALIDATION_RULES:
#         raise ValueError(f"No validation rules configured for: {table_name}")
#     rules = VALIDATION_RULES[table_name]

#     for rule_name, rule_config in rules.items():
#         df = df.withColumn(
#             rule_name + "_validation",
#             rule_config["condition"]
#         )

#     return df

In [0]:
def apply_validation_rules(table_name, df):

    validation_config = VALIDATION_RULES[table_name]

    # Start every record as VALID
    result_df = (
        df
        .withColumn("record_status", F.lit("VALID"))
        .withColumn(
            "validation_reason",
            F.lit(None).cast("string")
        )
    )

    # Apply every configured validation rule
    for rule_name, rule_config in validation_config.items():

        condition = rule_config["condition"]
        error_code = rule_config["error_code"]

        result_df = (
            result_df
            .withColumn(
                "record_status",
                F.when(
                    condition,
                    F.lit("INVALID")
                ).otherwise(
                    F.col("record_status")
                )
            )
            .withColumn(
                "validation_reason",
                F.when(
                    condition,
                    F.when(
                        F.col("validation_reason").isNull(),
                        F.lit(error_code)
                    ).otherwise(
                        F.concat(
                            F.col("validation_reason"),
                            F.lit(";"),
                            F.lit(error_code)
                        )
                    )
                ).otherwise(
                    F.col("validation_reason")
                )
            )
        )

    return result_df

###Generic record_status column

In [0]:
# def add_record_status(table_name, df):
#     rules = VALIDATION_RULES[table_name]
#     validation_columns = [
#         rule_name+"_validation" for rule_name in rules.keys()]
    
#     invalid_condition = F.lit(False)

#     for column in validation_columns:
#         invalid_condition = invalid_condition | F.col(column)


#     df = df.withColumn("record_status", F.when(invalid_condition, "INVALID").otherwise("VALID"))

#     return df

In [0]:
# def add_validation_reason(table_name, df):
#     rules = VALIDATION_RULES[table_name]
#     reason_array = []
#     for rule_name, rule_config in rules.items():

#         validation_column = (
#             rule_name + "_validation"
#         )

#         reason_array.append(
#             F.when(
#                 F.col(validation_column),
#                 F.lit(rule_config["error_code"])
#             )
#         )
#     df = df.withColumn(
#         "validation_errors",
#         F.array(*reason_array)
#     )

#     df = df.withColumn(
#         "validation_errors",
#         F.expr(
#             "filter(validation_errors, x -> x is not null)"
#         )
#     )
#     return df

###Generic deduplication function

In [0]:
def deduplicate_records(table_name, df):
    config = SILVER_CONFIG[table_name]
    primary_key = config["primary_key"]
    watermark_column = config["watermark_column"]

    window_spec = Window.partitionBy(primary_key).orderBy(F.desc(watermark_column))

    deduplicated_df = df.withColumn("_row_number", F.row_number().over(window_spec))\
        .filter(F.col("_row_number") == 1).drop("_row_number")

    return deduplicated_df

###Drop Validation Columns

In [0]:
def prepare_silver_output(table_name, df):
    rules = VALIDATION_RULES.get(table_name, {})

    validation_columns = [rules_name + "_validation" for rules_name in rules.keys()]

    columns_to_drop = validation_columns + ["validation_errors"]

    return df.drop(*columns_to_drop)

###Generic Merge Function

In [0]:
def merge_to_silver(source_df, target_table, primary_key):
    if source_df.limit(1).count() == 0:
        print(f"No records to MERGE into {target_table}")
        return
    target = DeltaTable.forName(spark, target_table)
    target.alias("t").merge(source_df.alias("s"),
                            f"t.{primary_key} = s.{primary_key}")\
                                .whenMatchedUpdateAll()\
                                    .whenNotMatchedInsertAll().execute()

    print(
        f"MERGE completed successfully: {target_table}"
    )

###Reusable write function

In [0]:
# def write_delta_table(table_name, df, mode="overwrite"):
#     df.write.format("delta").mode(mode).option("overwriteSchema","true").saveAsTable(table_name)

#     print(f"Successfully written to: {table_name}")

In [0]:
def write_to_quarantine(table_name, df):

    config = validate_table_config(table_name)
    quarantine_table = config["quarantine_table"]

    if df.limit(1).count() == 0:
        print("No invalid records to quarantine.")
        return

    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(quarantine_table)
    )

    print(f"Quarantine records written to: {quarantine_table}")

### Audit insertion

In [0]:
def log_silver_audit(run_id,
    table_name,
    start_time,
    end_time,
    status,
    source_count,
    valid_count,
    invalid_count,
    duplicate_count,
    silver_count,
    error_message=None,
    watermark_value=None):
    audit_schema = StructType([
    StructField(
        "run_id",
        StringType(),
        True
    ),
    StructField(
        "table_name",
        StringType(),
        True
    ),
    StructField(
        "start_time",
        TimestampType(),
        True
    ),
    StructField(
        "end_time",
        TimestampType(),
        True
    ),
    StructField(
        "status",
        StringType(),
        True
    ),
    StructField(
        "source_count",
        LongType(),
        True
    ),
    StructField(
        "valid_count",
        LongType(),
        True
    ),
    StructField(
        "invalid_count",
        LongType(),
        True
    ),
    StructField(
        "duplicate_count",
        LongType(),
        True
    ),
    StructField(
        "silver_count",
        LongType(),
        True
    ),
    StructField(
        "error_message",
        StringType(),
        True
    ),
    StructField(
        "watermark_value",
        TimestampType(),
        True
    )
    ])
    audit_data = [(run_id,
    table_name,
    start_time,
    end_time,
    status,
    source_count,
    valid_count,
    invalid_count,
    duplicate_count,
    silver_count,
    error_message, watermark_value)]
    audit_df = spark.createDataFrame(audit_data,audit_schema)

    audit_df.write.format("delta").mode("append").saveAsTable("quickcart.silver.silver_audit")

###Silver Framework

In [0]:
def process_to_silver(table_name):
    table_name = table_name.lower().strip()
    run_id = str(uuid.uuid4())
    start_time = datetime.now()

    print("=" * 70)
    print(f"Starting Silver processing: {table_name}")
    print(f"Run ID: {run_id}")
    print("=" * 70)

    try:
        #STEP 1: Validate Configuration
        config = validate_table_config(table_name)
        bronze_table = config["bronze_table"]
        silver_table = config["silver_table"]
        quarantine_table = config["quarantine_table"]
        primary_key = config["primary_key"]
        watermark_column = config["watermark_column"]

        print(f"Bronze Table      : {bronze_table}")
        print(f"Silver Table      : {silver_table}")
        print(f"Quarantine Table  : {quarantine_table}")
        print(f"Primary Key       : {primary_key}")
        print(f"Watermark Column  : {watermark_column}")

        #STEP 2: Read Bronze 
        bronze_df = spark.table(bronze_table)

        print(
            f"Bronze Record Count: {bronze_df.count()}"
        )       

        #STEP 3: Get last watermark
        last_watermark = get_last_watermark(table_name)

        print(f"Last watermark: {last_watermark}")

        #STEP 4: Determine FULL vs INCREMENTAL
        if last_watermark is None:
            print("No previous watermark found.")
            print("Processing FULL LOAD.")
            processing_df = bronze_df
            load_type = "FULL"
        else:
            print("Previous watermark found.")
            print("Processing INCREMENTAL LOAD.")
            processing_df = (
                bronze_df
                .filter(
                    F.col(watermark_column) > F.lit(last_watermark)
                )
            )
            load_type = "INCREMENTAL"

        #STEP 5: Count processing records
        source_count = processing_df.count()

        print(f"Load type: {load_type}")
        print(f"Records to process: {source_count}")

        #STEP 6: Handle zero records
        if source_count == 0:

            print("No new records to process.")

            end_time = datetime.now()

            log_silver_audit(
                run_id=run_id,
                table_name=table_name,
                start_time=start_time,
                end_time=end_time,
                status="SUCCESS",
                source_count=0,
                valid_count=0,
                invalid_count=0,
                duplicate_count=0,
                silver_count=spark.table(silver_table).count(),
                error_message=None,
                watermark_value=last_watermark
            )

            print("Silver processing completed - no new records.")

            return {
                "run_id": run_id,
                "table_name": table_name,
                "status": "SUCCESS",
                "load_type": load_type,
                "source_count": 0,
                "valid_count": 0,
                "invalid_count": 0,
                "duplicate_count": 0,
                "silver_count": spark.table(silver_table).count(),
                "watermark_value": last_watermark
            }
        
        #STEP 7: Calculate new watermark
        watermark_value = processing_df\
            .agg(
                F.max(watermark_column).alias("max_watermark")
            ).first()["max_watermark"]  

        print(f"New watermark: {watermark_value}")

        # STEP 8: Standardize columns
        standardized_df = standardize_columns(table_name,processing_df)
        print("Standardization completed successfully.")

        #STEP 9: Apply validation rules
        validated_df = apply_validation_rules(table_name, standardized_df)
        print("Validation completed successfully.")
        print("Added Record status column")
        print("Added Validation Reason column")

        # #STEP 10: Add record status
        # validated_df = add_record_status(table_name, validated_df)
        # print("Added Record status column")

        # #STEP 11: Add validation reason
        # validated_df = add_validation_reason(table_name, validated_df)
        # print("Added Validation Reason column")

        #STEP 12: Split VALID / INVALID
        valid_df = validated_df.filter(col('record_status') == "VALID")
        print("Valid:", valid_df.count())
        before_count = valid_df.count()

        invalid_df = validated_df.filter(col('record_status') == "INVALID")
        print("Invalid:", invalid_df.count())
        invalid_count = invalid_df.count()

        #STEP 13: Write invalid records to quarantine
        write_to_quarantine(table_name, invalid_df)

        #STEP 14: Deduplicate valid records
        valid_deduplicated_df = deduplicate_records(table_name, valid_df)
        after_count = valid_deduplicated_df.count()

        duplicate_count = before_count - after_count

        print(f"Valid records before deduplication : {before_count}")
        print(f"Valid records after deduplication  : {after_count}")
        print(f"Duplicate records removed          : {duplicate_count}")

        #STEP 15: Prepare Silver output
        silver_output_df = prepare_silver_output(table_name, valid_deduplicated_df)

        #STEP 16: Write to Silver
        if load_type == "FULL":
            (
                silver_output_df.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(silver_table)
            )

            print(f"FULL LOAD completed: {silver_table}")

        else:
            merge_to_silver(silver_output_df,silver_table,primary_key)
        
        #STEP 17: Get final Silver count
        silver_count = spark.table(silver_table).count()

        print(f"Silver record count: {silver_count}")

        #STEP 18: Audit SUCCESS
        end_time = datetime.now()

        log_silver_audit(
            run_id=run_id,
            table_name=table_name,
            start_time=start_time,
            end_time=end_time,
            status="SUCCESS",
            source_count=source_count,
            valid_count=before_count,
            invalid_count=invalid_count,
            duplicate_count=duplicate_count,
            silver_count=after_count,
            error_message=None,
            watermark_value=watermark_value
        )

        #STEP 19: Return metrics
        print("=" * 60)
        print("Silver processing completed successfully.")
        print("=" * 60)
        return {
            "run_id": run_id,
            "table_name": table_name,
            "status": "SUCCESS",
            "load_type": load_type,
            "source_count": source_count,
            "valid_count": before_count,
            "invalid_count": invalid_count,
            "duplicate_count": duplicate_count,
            "silver_count": after_count,
            "watermark_value": watermark_value
         }
    except Exception as e:
        #ERROR HANDLING
         end_time = datetime.now()
         error_message = str(e)
         print("=" * 60)
         print("Silver processing FAILED")
         print(f"Error: {error_message}")
         print("=" * 60)
         log_silver_audit(
            run_id=run_id,
            table_name=table_name,
            start_time=start_time,
            end_time=end_time,
            status="FAILED",
            source_count=0,
            valid_count=0,
            invalid_count=0,
            duplicate_count=0,
            silver_count=0,
            error_message=error_message,
            watermark_value=None
         )
         return{
             "run_id" : run_id,
             "table_name": table_name,
             "status": "FAILED",
             "load_type": None,
            "source_count": 0,
            "valid_count": 0,
            "invalid_count": 0,
            "duplicate_count": 0,
            "silver_count": 0,
            "watermark_value": None,
            "error_message": str(e)
        }         

In [0]:
result = process_to_silver("deliveries")
print(result)

# if result["status"] == "SUCCESS":
#     display(
#         spark.table("quickcart.silver.customers").limit(10)
#     )

In [0]:
%sql
SELECT COUNT(*) FROM quickcart.silver.customers;
SELECT COUNT(*) FROM quickcart.silver.products;
SELECT COUNT(*) FROM quickcart.silver.orders;
SELECT COUNT(*) FROM quickcart.silver.payments;
SELECT COUNT(*) FROM quickcart.silver.deliveries;

In [0]:
%sql
SELECT * FROM quickcart.silver.silver_audit;

#TEST

In [0]:
last_watermark = get_last_watermark(
    "customers"
)

print(
    "Last Watermark:",
    last_watermark
)

In [0]:
# customers_bronze = spark.table(
#     "quickcart.bronze.customers"
# )

# # customers_bronze.count()



# customers_incremental = get_incremental_data("customers", customers_bronze)
# customers_incremental.count()

# standardized_df = standardize_columns("customers",customers_incremental)
# print("Standardization completed successfully.")

# validated_df = apply_validation_rules("customers", standardized_df)
# print("Validation completed successfully.")

# validated_df = add_record_status("customers", validated_df)
# print("Added Record status")

# validated_df = add_validation_reason("customers", validated_df)
# print("Added Validation Reason")

# valid_df = validated_df.filter(col('record_status') == "VALID")
# print("Valid:", valid_df.count())

# invalid_df = validated_df.filter(col('record_status') == "INVALID")
# print("Invalid:", invalid_df.count())

# valid_deduplicated_df = deduplicate_records("customers", valid_df)
# print("Deduplication Completed:", valid_deduplicated_df.count())

# customers_incremental_valid = prepare_silver_output("customers", valid_deduplicated_df)
# print("incremental validation Completed:", customers_incremental_valid.count())

# config = validate_table_config(
#     "customers"
# )

# primary_key = config["primary_key"]
# silver_table = config["silver_table"]







In [0]:
# merge_to_silver(
#     customers_incremental_valid,
#     silver_table,
#     primary_key
# )

In [0]:
display(
    spark.table(
        "quickcart.silver.customers"
    )
    .filter(
        F.col("customer_id").startswith("INC_")
    )
)

In [0]:
# customers_bronze.select(
#     F.max("updated_at").alias("max_updated_at")
# ).show()

####Schemas for other tables

In [0]:
for table in [
    "products",
    "orders",
    "payments",
    "deliveries"
]:
    print(f"\n{'='*60}")
    print(f"{table.upper()}")
    print(f"{'='*60}")

    display(spark.table(
        f"quickcart.bronze.{table}"
    ).limit(1))

In [0]:
# Deliveries
display(
    spark.table("quickcart.bronze.deliveries")
    .select("delivery_partner")
    .distinct()
)

In [0]:
for table_name in [
    "customers",
    "products",
    "orders",
    "payments",
    "deliveries"
]:

    print(f"\n===== {table_name.upper()} =====")

    bronze_df = spark.table(
        SILVER_CONFIG[table_name]["bronze_table"]
    )

    validation_df = apply_validation_rules(
        table_name, bronze_df
    )

    validation_df.groupBy(
        "record_status"
    ).count().show()

###Record the baseline watermark

In [0]:
# silver_count = spark.table(
#     "quickcart.silver.customers"
# ).count()

# baseline_run_id = f"BASELINE-{uuid.uuid4()}"

# baseline_audit_data = [(
#     baseline_run_id,
#     "customers",
#     datetime.now(),
#     datetime.now(),
#     "SUCCESS",
#     200030,
#     silver_count,
#     0,
#     0,
#     silver_count,
#     None,
#     baseline_watermark
# )]

# audit_schema = StructType([
#     StructField(
#         "run_id",
#         StringType(),
#         True
#     ),
#     StructField(
#         "table_name",
#         StringType(),
#         True
#     ),
#     StructField(
#         "start_time",
#         TimestampType(),
#         True
#     ),
#     StructField(
#         "end_time",
#         TimestampType(),
#         True
#     ),
#     StructField(
#         "status",
#         StringType(),
#         True
#     ),
#     StructField(
#         "source_count",
#         LongType(),
#         True
#     ),
#     StructField(
#         "valid_count",
#         LongType(),
#         True
#     ),
#     StructField(
#         "invalid_count",
#         LongType(),
#         True
#     ),
#     StructField(
#         "duplicate_count",
#         LongType(),
#         True
#     ),
#     StructField(
#         "silver_count",
#         LongType(),
#         True
#     ),
#     StructField(
#         "error_message",
#         StringType(),
#         True
#     ),
#     StructField(
#         "watermark_value",
#         TimestampType(),
#         True
#     )
#     ])

# baseline_audit_df = spark.createDataFrame(
#     baseline_audit_data,
#     audit_schema
# )

# (
#     baseline_audit_df.write
#     .format("delta")
#     .mode("append")
#     .saveAsTable("quickcart.silver.silver_audit")
# )